## 10 - Update Database with All 75 Transcripts

This notebook adds all new transcripts to the database.
Our existing 14 transcripts stay untouched - we only add new ones.

### What we add?
- 61 new transcripts across 31 new companies
- 2 market periods per company: Q1 2020 and Q2 2023
- Total after update: 75 transcripts in database

In [1]:
# Import libraries
import sqlite3
import pandas as pd
import os

# Connect to database
conn = sqlite3.connect("../data/earnings_research.db")

# Check current state
current = pd.read_sql("SELECT COUNT(*) as count FROM transcripts", conn)
print(f"Current transcripts in database: {current['count'][0]}")

# Get all clean transcript files
transcripts_folder = "../data/transcripts"
all_clean = sorted([f for f in os.listdir(transcripts_folder) if f.endswith("_clean.txt")])
print(f"Clean transcript files available: {len(all_clean)}")

# Get already loaded transcripts
already_loaded = pd.read_sql("""
    SELECT ticker || '_' || quarter || '_clean.txt' as filename
    FROM transcripts
""", conn)

already_loaded_list = already_loaded["filename"].tolist()
print(f"Already in database: {len(already_loaded_list)}")

# Find new ones to load
new_transcripts = [f for f in all_clean if f not in already_loaded_list]
print(f"New transcripts to load: {len(new_transcripts)}")

Current transcripts in database: 14
Clean transcript files available: 75
Already in database: 14
New transcripts to load: 61


In [2]:
# Load all 61 new transcripts into database
success = []
failed = []

for filename in new_transcripts:
    # Parse ticker and quarter from filename
    # Format: TICKER_Q#_YEAR_clean.txt
    parts = filename.replace("_clean.txt", "").split("_")
    ticker = parts[0]
    quarter = parts[1] + "_" + parts[2]
    year = parts[2]
    
    # Assign period and date based on quarter
    if "2020" in quarter and "Q1" in quarter:
        period = "Pre-COVID"
        date = "2020-04-01"  # approximate Q1 2020
    elif "2020" in quarter and "Q2" in quarter:
        period = "COVID Crash"
        date = "2020-07-01"
    elif "2021" in quarter:
        period = "Recovery"
        date = "2021-07-01"
    elif "2022" in quarter:
        period = "Rate Hike Peak"
        date = "2022-10-01"
    elif "2023" in quarter and "Q2" in quarter:
        period = "AI Boom"
        date = "2023-08-01"
    elif "2023" in quarter and "Q1" in quarter:
        period = "AI Boom"
        date = "2023-04-01"
    elif "2025" in quarter:
        period = "Tariff Shock"
        date = "2025-04-01"
    else:
        period = "Unknown"
        date = "2020-01-01"
    
    # Read clean transcript
    try:
        filepath = f"{transcripts_folder}/{filename}"
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()
        
        # Insert into database
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO transcripts (ticker, quarter, date, period, raw_text)
            VALUES (?, ?, ?, ?, ?)
        """, (ticker, quarter, date, period, text))
        
        success.append(filename)
        print(f"✓ {filename}")
    
    except Exception as e:
        failed.append(filename)
        print(f"✗ {filename}: {e}")

conn.commit()
print(f"\nSuccessfully loaded: {len(success)}")
print(f"Failed: {len(failed)}")

✓ ABT_Q1_2020_clean.txt
✓ ABT_Q2_2023_clean.txt
✓ AMD_Q1_2020_clean.txt
✓ AMD_Q2_2023_clean.txt
✓ BAC_Q1_2020_clean.txt
✓ BAC_Q2_2023_clean.txt
✓ BLK_Q1_2020_clean.txt
✓ BLK_Q2_2023_clean.txt
✓ COST_Q1_2020_clean.txt
✓ COST_Q2_2023_clean.txt
✓ CRM_Q1_2020_clean.txt
✓ CRM_Q2_2023_clean.txt
✓ CVX_Q1_2020_clean.txt
✓ CVX_Q2_2023_clean.txt
✓ DHR_Q1_2020_clean.txt
✓ DHR_Q2_2023_clean.txt
✓ GOOGL_Q1_2020_clean.txt
✓ GOOGL_Q2_2023_clean.txt
✓ GS_Q1_2020_clean.txt
✓ GS_Q2_2023_clean.txt
✓ HAL_Q1_2020_clean.txt
✓ HAL_Q2_2023_clean.txt
✓ HD_Q1_2020_clean.txt
✓ HD_Q2_2023_clean.txt
✓ INTC_Q1_2020_clean.txt
✓ INTC_Q2_2023_clean.txt
✓ KO_Q1_2020_clean.txt
✓ KO_Q2_2023_clean.txt
✓ MCD_Q1_2020_clean.txt
✓ MCD_Q2_2023_clean.txt
✓ META_Q1_2020_clean.txt
✓ META_Q2_2023_clean.txt
✓ MRK_Q1_2020_clean.txt
✓ MRK_Q2_2023_clean.txt
✓ MS_Q1_2020_clean.txt
✓ MS_Q2_2023_clean.txt
✓ NKE_Q1_2020_clean.txt
✓ NKE_Q2_2023_clean.txt
✓ OXY_Q1_2020_clean.txt
✓ OXY_Q2_2023_clean.txt
✓ PFE_Q1_2020_clean.txt
✓ PFE_Q2_2023_

In [3]:
# Verify final database state
total = pd.read_sql("SELECT COUNT(*) as count FROM transcripts", conn)
print(f"Total transcripts in database: {total['count'][0]}")

# Count by period
by_period = pd.read_sql("""
    SELECT period, COUNT(*) as count
    FROM transcripts
    GROUP BY period
    ORDER BY count DESC
""", conn)
print("\nTranscripts by market period:")
print(by_period.to_string(index=False))

# Count by ticker
by_ticker = pd.read_sql("""
    SELECT ticker, COUNT(*) as count
    FROM transcripts
    GROUP BY ticker
    ORDER BY ticker
""", conn)
print(f"\nCompanies covered: {len(by_ticker)}")
print(by_ticker.to_string(index=False))

# Close connection
conn.close()
print("\nDatabase connection closed.")

Total transcripts in database: 75

Transcripts by market period:
        period  count
     Pre-COVID     36
       AI Boom     32
  Tariff Shock      2
Rate Hike Peak      2
   COVID Crash      2
      Recovery      1

Companies covered: 37
ticker  count
  AAPL      4
   ABT      2
   AMD      2
  AMZN      2
   BAC      2
   BLK      2
  COST      2
   CRM      2
   CVX      2
   DHR      2
 GOOGL      2
    GS      2
   HAL      2
    HD      2
  INTC      2
   JNJ      1
   JPM      2
    KO      2
   MCD      2
  META      2
   MRK      2
    MS      2
  MSFT      3
   NKE      2
  NVDA      1
   OXY      2
   PFE      2
    PG      2
  SBUX      2
   SLB      3
   TGT      2
   TMO      2
   UNH      2
   VLO      2
   WFC      2
   WMT      2
   XOM      1

Database connection closed.
